# V-MD3 FMCW Range, Angle, and Rail-SAR Processing

This notebook processes the V-MD3 data collected during Runs A–D of the rail-SAR proof-of-concept experiment.

The notebook is organized into the following major sections:

1. **Notebook setup**
   - Imports
   - Repository and data directories
   - Experiment constants
   - Binary-file discovery
   - Binary decoding functions
   - Cube construction and validation functions
   - Common plotting and processing functions

2. **Run A: Conventional radar processing and validation**
   - Decode RADC and FPGA RFFT binary data
   - Validate array dimensions and axis meanings
   - Construct one-dimensional range profiles
   - Construct two-dimensional range–azimuth heatmaps
   - Compare offline RADC processing with the FPGA RFFT output
   - Evaluate frame-to-frame phase stability

3. **Runs B and C: Centered-target SAR**
   - Use Run B as the background dataset
   - Use Run C as the centered-sphere dataset
   - Construct the coherent SAR image
   - Verify the reconstructed target location

4. **Runs B and D: Offset-target SAR**
   - Use Run B as the background dataset
   - Use Run D as the offset-sphere dataset
   - Reconstruct the SAR image
   - Verify that the target shifts in the expected cross-range direction

All reusable decoding, validation, signal-processing, and plotting operations should be defined as functions in the setup section rather than duplicated throughout the notebook.

## Experiment Data Organization

The experiment contains four acquisition runs.

### Run A — Fixed-position phase test

- The radar remains at the center rail position.
- The sphere is present near boresight.
- Approximately 20 repeated frames were recorded.
- Run A does **not** contain a 17-position synthetic aperture.
- Run A is used to validate:
  - binary decoding;
  - offline range processing;
  - FPGA RFFT interpretation;
  - range–azimuth processing;
  - frame-to-frame phase stability.

### Run B — Background rail scan

- The sphere is removed.
- Data are recorded at 17 mechanical positions.
- Run B characterizes static clutter from the rail, mounting structure, absorber, floor, cables, and surrounding environment.

### Run C — Centered-sphere rail scan

- The sphere is located near the center of the synthetic-aperture scene.
- Data are recorded at 17 mechanical positions.
- Position index 8 is the center rail position.

### Run D — Offset-sphere rail scan

- The sphere is displaced in cross-range relative to Run C.
- Data are recorded at the same 17 mechanical positions.
- Run D is used to verify that the reconstructed target moves in the expected direction.

The SAR position indices are

$$p = 0,1,\ldots,16,$$

with

$$p_{\mathrm{center}} = 8.$$

## 1.0 Notebook Setup

This section establishes the reusable infrastructure used throughout the notebook.

The setup section contains:

1. **Python imports, repository paths, and experiment definitions**
   - repository and data directories;
   - Run A–D binary-data directories;
   - SAR position indices;
   - center-position index;
   - Galil encoder counts.

2. **Binary-file discovery**
   - locate all `.bin` files for Runs A–D;
   - verify the expected number of files in each run.

3. **Reusable function definitions**
   - filename and position-index parsing;
   - binary-record parsing;
   - payload reading;
   - RADC decoding;
   - FPGA RFFT decoding;
   - DONE decoding;
   - stream selection and stack loading;
   - array-shape and data-integrity validation.

4. **File-level manifest construction and validation**

   `FILE_MANIFEST` contains one row per acquisition file, including:

   - run identifier;
   - acquisition type;
   - position index, where applicable;
   - Galil encoder count, where applicable;
   - filename;
   - file path;
   - file size.

5. **Record-level manifest construction and validation**

   `RECORD_MANIFEST` contains one row per stored V-MD3 record, including:

   - run identifier;
   - acquisition file;
   - position index and encoder count, where applicable;
   - record index within the file;
   - stream type: `RADC`, `RFFT`, or `DONE`;
   - frame index within each stream;
   - header and payload byte offsets;
   - payload length;
   - decoded DONE value, where applicable.

6. **Run A loading and validation**
   - load all Run A RADC frames;
   - load all Run A FPGA RFFT frames;
   - load the Run A DONE values;
   - verify frame counts, cube shapes, finite values, and DONE-counter continuity.

The binary files are indexed during setup, but the complete Runs B–D datasets will generally be decoded only when required by their respective processing sections. Run A is loaded during setup because it is used immediately in the next section.

This keeps the notebook self-contained while avoiding unnecessary memory use and long startup times.

The intended data flow is:

```text
binary files
    ↓
file-level manifest
    ↓
record-level manifest
    ↓
stream-specific payload decoder
    ↓
validated NumPy stack
    ↓
range, angle, phase, and SAR processing
```

### Canonical Array Shapes

One decoded RADC frame has shape:

```text
RADC frame: (128, 64, 4)
```

with axis meanings:

```text
axis 0 = fast-time ADC sample
axis 1 = chirp / slow-time index
axis 2 = receive-channel index
```

Therefore:

```python
radc_cube[sample, chirp, channel]
```

A stack of RADC frames has shape:

```text
RADC stack: (number_of_frames, 128, 64, 4)
```

with representation:

```python
radc_stack[frame, sample, chirp, channel]
```

One decoded FPGA RFFT frame also has shape:

```text
RFFT frame: (128, 64, 4)
```

and is provisionally represented as:

```python
rfft_cube[range_bin, slow_time_index, channel]
```

A stack of FPGA RFFT frames has shape:

```text
RFFT stack: (number_of_frames, 128, 64, 4)
```

with representation:

```python
rfft_stack[frame, range_bin, slow_time_index, channel]
```

The term `slow_time_index` is provisional. Run A processing will determine whether this axis represents:

- chirp-indexed range-FFT outputs;
- Doppler-bin outputs;
- or another FPGA-specific ordering.

After its meaning is confirmed, the corresponding variable names, axis labels, comments, and docstrings should be updated throughout the notebook.

### 1.1 Python Imports and Repository and data-directory definitions

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

In [ ]:
# ---------------------------------------------------------------------
# Repository and data locations
# ---------------------------------------------------------------------

REPO_ROOT = Path(r"D:\Research\repos\vmd3_projects")
DATA_ROOT = REPO_ROOT / "data"

RUN_DIRS = {
    letter: DATA_ROOT / f"20260709_run_{letter}" / "bins"
    for letter in "abcd"
}

# ---------------------------------------------------------------------
# SAR acquisition positions: Runs B, C, and D
# ---------------------------------------------------------------------

SAR_POSITION_INDICES = np.arange(17)
CENTER_POSITION_INDEX = 8

GALIL_ENCODER_COUNTS = np.arange(
    0,
    800_000 + 50_000,
    50_000,
)

assert len(SAR_POSITION_INDICES) == 17
assert len(GALIL_ENCODER_COUNTS) == 17
assert SAR_POSITION_INDICES[CENTER_POSITION_INDEX] == 8
assert GALIL_ENCODER_COUNTS[CENTER_POSITION_INDEX] == 400_000

print("Repository:", REPO_ROOT)
print()

for run, folder in RUN_DIRS.items():
    print(f"Run {run.upper()}: {folder}")
    print(f"  Exists: {folder.exists()}")

### 1.2 Binary-file discover to `BIN_FILES`


In [ ]:
# ---------------------------------------------------------------------
# Inspect available binary files
# ---------------------------------------------------------------------

BIN_FILES = {}

for run, folder in RUN_DIRS.items():
    files = sorted(folder.glob("*.bin"))
    BIN_FILES[run] = files

    print(f"Run {run.upper()}")
    print(f"  Folder: {folder}")
    print(f"  Number of .bin files: {len(files)}")

    if not files:
        print("  No binary files found.")
    else:
        for path in files:
            size_bytes = path.stat().st_size
            print(f"  {path.name:<60} {size_bytes:>12,} bytes")

    print()

### 1.3 Function Definitions 
All binary parsing, decoding, loading, and validation functions

In [ ]:
# ---------------------------------------------------------------------
# Build and validate the binary-file manifest
# ---------------------------------------------------------------------

_POSITION_PATTERN = re.compile(
    r"^scan_(?:pos)?(?P<position>\d+)cm\.bin$",
    flags=re.IGNORECASE,
)


def parse_sar_position_index(filename: str) -> int:
    """
    Extract the acquisition position index from a Run B, C, or D filename.

    Examples
    --------
    scan_0cm.bin     -> 0
    scan_pos8cm.bin  -> 8
    scan_pos16cm.bin -> 16

    The extracted number is treated as a position index, not as a
    trustworthy physical distance in centimeters.
    """
    match = _POSITION_PATTERN.fullmatch(filename)

    if match is None:
        raise ValueError(
            f"Could not determine position index from filename: {filename}"
        )

    return int(match.group("position"))

# ---------------------------------------------------------------------
# V-MD3 binary parsing, decoding, loading, and validation functions
# ---------------------------------------------------------------------

FRAME_HEADER_BYTES = 8

STREAM_CODE_TO_NAME = {
    b"RADC": "radc",
    b"RFFT": "rfft",
    b"DONE": "done",
}

EXPECTED_PAYLOAD_BYTES = {
    "radc": 131_072,
    "rfft": 131_072,
    "done": 4,
}

RADC_FRAME_SHAPE = (128, 64, 4)
RFFT_FRAME_SHAPE = (128, 64, 4)


def scan_vmd3_bin(path: Path) -> pd.DataFrame:
    """
    Scan one acquisition .bin file and return one row per stored frame.

    Stored frame format
    -------------------
    bytes 0:4   Stream code: RADC, RFFT, or DONE
    bytes 4:8   Payload length, little-endian unsigned integer
    bytes 8:    Payload
    """
    path = Path(path)
    file_size = path.stat().st_size

    rows = []
    stream_counts = {
        stream_name: 0
        for stream_name in STREAM_CODE_TO_NAME.values()
    }

    offset = 0
    record_index = 0

    with path.open("rb") as file:
        while offset < file_size:
            file.seek(offset)
            header = file.read(FRAME_HEADER_BYTES)

            if len(header) != FRAME_HEADER_BYTES:
                raise EOFError(
                    f"{path.name}: incomplete frame header "
                    f"at byte offset {offset:,}."
                )

            stream_code = header[:4]

            if stream_code not in STREAM_CODE_TO_NAME:
                raise ValueError(
                    f"{path.name}: unknown stream code "
                    f"{stream_code!r} at byte offset {offset:,}."
                )

            stream = STREAM_CODE_TO_NAME[stream_code]

            payload_length = int.from_bytes(
                header[4:8],
                byteorder="little",
                signed=False,
            )

            expected_length = EXPECTED_PAYLOAD_BYTES[stream]

            if payload_length != expected_length:
                raise ValueError(
                    f"{path.name}: {stream.upper()} record "
                    f"{record_index} has payload length "
                    f"{payload_length:,}; expected "
                    f"{expected_length:,}."
                )

            payload_offset = offset + FRAME_HEADER_BYTES
            next_offset = payload_offset + payload_length

            if next_offset > file_size:
                raise EOFError(
                    f"{path.name}: record {record_index} extends "
                    "beyond the end of the file."
                )

            stream_frame_index = stream_counts[stream]
            stream_counts[stream] += 1

            rows.append(
                {
                    "record_index": record_index,
                    "stream": stream,
                    "stream_frame_index": stream_frame_index,
                    "header_offset": offset,
                    "payload_offset": payload_offset,
                    "payload_length": payload_length,
                    "frame_length": (
                        FRAME_HEADER_BYTES + payload_length
                    ),
                }
            )

            offset = next_offset
            record_index += 1

    if offset != file_size:
        raise ValueError(
            f"{path.name}: parser stopped at byte {offset:,}, "
            f"but the file size is {file_size:,}."
        )

    return pd.DataFrame(rows)


def read_record_payload(
    path: Path,
    payload_offset: int,
    payload_length: int,
) -> bytes:
    """Read one payload from a parsed V-MD3 record."""
    path = Path(path)

    with path.open("rb") as file:
        file.seek(int(payload_offset))
        payload = file.read(int(payload_length))

    if len(payload) != int(payload_length):
        raise EOFError(
            f"{path.name}: requested {payload_length:,} bytes "
            f"at offset {payload_offset:,}, but read "
            f"{len(payload):,} bytes."
        )

    return payload


def decode_radc_2d(payload: bytes) -> np.ndarray:
    """
    Decode one mode-0 RADC payload.

    Returns
    -------
    cube
        Shape: (sample, chirp, channel) = (128, 64, 4)
    """
    expected_bytes = EXPECTED_PAYLOAD_BYTES["radc"]

    if len(payload) != expected_bytes:
        raise ValueError(
            f"Expected {expected_bytes:,} RADC bytes, "
            f"received {len(payload):,}."
        )

    # Stored as:
    # chirp, channel, interleaved [Q0, I0, Q1, I1, ...]
    raw = np.frombuffer(
        payload,
        dtype="<i2",
    ).reshape(64, 4, 256)

    q = raw[:, :, 0::2].astype(np.float64)
    i = raw[:, :, 1::2].astype(np.float64)

    # Convert from (chirp, channel, sample)
    # to (sample, chirp, channel).
    cube = np.transpose(
        i + 1j * q,
        (2, 0, 1),
    )

    if cube.shape != RADC_FRAME_SHAPE:
        raise ValueError(
            f"Decoded RADC shape is {cube.shape}; "
            f"expected {RADC_FRAME_SHAPE}."
        )

    return cube


def decode_rfft_2d(payload: bytes) -> np.ndarray:
    """
    Decode one mode-0 FPGA RFFT payload.

    Returns
    -------
    cube
        Provisional shape:
        (range_bin, slow_time_index, channel) = (128, 64, 4)

    Notes
    -----
    The meaning of slow_time_index is not yet confirmed. It may represent
    chirp/slow time, Doppler bins, or another FPGA-specific ordering.
    """
    expected_bytes = EXPECTED_PAYLOAD_BYTES["rfft"]

    if len(payload) != expected_bytes:
        raise ValueError(
            f"Expected {expected_bytes:,} RFFT bytes, "
            f"received {len(payload):,}."
        )

    # Provisional FPGA storage arrangement:
    # channel, range_bin, axis1_index, [Q, I]
    raw = np.frombuffer(
        payload,
        dtype="<i2",
    ).reshape(4, 128, 64, 2)

    q = raw[:, :, :, 0].astype(np.float64)
    i = raw[:, :, :, 1].astype(np.float64)

    # Convert to:
    # range_bin, axis1_index, channel
    cube = np.transpose(
        i + 1j * q,
        (1, 2, 0),
    )

    if cube.shape != RFFT_FRAME_SHAPE:
        raise ValueError(
            f"Decoded RFFT shape is {cube.shape}; "
            f"expected {RFFT_FRAME_SHAPE}."
        )

    return cube


def decode_done(payload: bytes) -> int:
    """
    Decode one DONE payload as a little-endian unsigned integer.

    The values appear to behave as radar-side frame counters.
    """
    expected_bytes = EXPECTED_PAYLOAD_BYTES["done"]

    if len(payload) != expected_bytes:
        raise ValueError(
            f"Expected {expected_bytes} DONE bytes, "
            f"received {len(payload)}."
        )

    return int.from_bytes(
        payload,
        byteorder="little",
        signed=False,
    )


def _select_stream_records(
    record_manifest: pd.DataFrame,
    run: str,
    stream: str,
    position_index: int | None = None,
) -> pd.DataFrame:
    """Select and order records for one run, stream, and position."""
    run = run.upper()
    stream = stream.lower()

    if run not in {"A", "B", "C", "D"}:
        raise ValueError(
            f"Unknown run {run!r}. Expected A, B, C, or D."
        )

    if stream not in {"radc", "rfft", "done"}:
        raise ValueError(
            f"Unknown stream {stream!r}."
        )

    records = record_manifest.loc[
        (record_manifest["run"] == run)
        & (record_manifest["stream"] == stream)
    ].copy()

    if run == "A":
        if position_index is not None:
            raise ValueError(
                "Run A is fixed-position and does not use "
                "a SAR position index."
            )
    else:
        if position_index is None:
            raise ValueError(
                f"position_index is required for Run {run}."
            )

        position_index = int(position_index)

        if position_index not in SAR_POSITION_INDICES:
            raise ValueError(
                f"Position index {position_index} is outside "
                "the expected range 0 through 16."
            )

        records = records.loc[
            records["position_index"] == position_index
        ]

    records = records.sort_values("stream_frame_index")

    if records.empty:
        raise ValueError(
            f"No {stream.upper()} records found for Run {run}, "
            f"position {position_index}."
        )

    return records


def load_radc_stack(
    record_manifest: pd.DataFrame,
    run: str,
    position_index: int | None = None,
) -> np.ndarray:
    """
    Load RADC data with shape:

        (frame, sample, chirp, channel)
    """
    records = _select_stream_records(
        record_manifest,
        run,
        "radc",
        position_index,
    )

    cubes = []

    for record in records.itertuples(index=False):
        payload = read_record_payload(
            record.path,
            record.payload_offset,
            record.payload_length,
        )

        cubes.append(decode_radc_2d(payload))

    return np.stack(cubes, axis=0)


def load_rfft_stack(
    record_manifest: pd.DataFrame,
    run: str,
    position_index: int | None = None,
) -> np.ndarray:
    """
    Load FPGA RFFT data with provisional shape:

        (frame, range_bin, slow_time_index, channel)
    """
    records = _select_stream_records(
        record_manifest,
        run,
        "rfft",
        position_index,
    )

    cubes = []

    for record in records.itertuples(index=False):
        payload = read_record_payload(
            record.path,
            record.payload_offset,
            record.payload_length,
        )

        cubes.append(decode_rfft_2d(payload))

    return np.stack(cubes, axis=0)


def load_done_values(
    record_manifest: pd.DataFrame,
    run: str,
    position_index: int | None = None,
) -> np.ndarray:
    """Load the DONE values for one run or SAR position."""
    records = _select_stream_records(
        record_manifest,
        run,
        "done",
        position_index,
    )

    values = []

    for record in records.itertuples(index=False):
        payload = read_record_payload(
            record.path,
            record.payload_offset,
            record.payload_length,
        )

        values.append(decode_done(payload))

    return np.asarray(values, dtype=np.uint32)


def validate_complex_stack(
    stack: np.ndarray,
    expected_frame_shape: tuple[int, int, int],
    expected_frames: int | None = None,
    label: str = "stack",
) -> None:
    """Validate one decoded complex-data stack."""
    if not isinstance(stack, np.ndarray):
        raise TypeError(f"{label} must be a NumPy array.")

    if stack.ndim != 4:
        raise ValueError(
            f"{label} must have four dimensions; "
            f"received shape {stack.shape}."
        )

    if stack.shape[1:] != expected_frame_shape:
        raise ValueError(
            f"{label} has per-frame shape {stack.shape[1:]}; "
            f"expected {expected_frame_shape}."
        )

    if (
        expected_frames is not None
        and stack.shape[0] != expected_frames
    ):
        raise ValueError(
            f"{label}: expected {expected_frames} frames, "
            f"received {stack.shape[0]}."
        )

    if not np.all(np.isfinite(stack)):
        raise ValueError(
            f"{label} contains NaN or infinite values."
        )

    print(f"{label} validation passed.")
    print(f"  Shape:               {stack.shape}")
    print(f"  Data type:           {stack.dtype}")
    print(f"  I minimum:           {stack.real.min():.0f}")
    print(f"  I maximum:           {stack.real.max():.0f}")
    print(f"  Q minimum:           {stack.imag.min():.0f}")
    print(f"  Q maximum:           {stack.imag.max():.0f}")
    print(
        "  Exact-zero fraction: "
        f"{np.mean(stack == 0):.6f}"
    )


def validate_record_manifest(
    record_manifest: pd.DataFrame,
) -> None:
    """
    Validate the stored records in every acquisition file.

    The acquisition code saves valid RADC, RFFT, and DONE frames in
    arrival order. Therefore, this function does not require repeating
    RADC-RFFT-DONE triplets.
    """
    expected_frames_per_file = {
        "A": 5,
        "B": 20,
        "C": 20,
        "D": 20,
    }

    expected_streams = ("radc", "rfft", "done")

    for (run, filename), group in record_manifest.groupby(
        ["run", "filename"],
        sort=False,
    ):
        group = group.sort_values("record_index").reset_index(drop=True)

        expected_count = expected_frames_per_file[run]
        expected_total_records = expected_count * len(expected_streams)

        # Check total number of stored records.
        if len(group) != expected_total_records:
            raise ValueError(
                f"Run {run}, {filename}: expected "
                f"{expected_total_records} total records, "
                f"found {len(group)}."
            )

        # Check that record indices are sequential.
        actual_record_indices = group["record_index"].astype(int).to_numpy()
        expected_record_indices = np.arange(expected_total_records)

        if not np.array_equal(
            actual_record_indices,
            expected_record_indices,
        ):
            raise ValueError(
                f"Run {run}, {filename}: record indices are "
                "not consecutive."
            )

        # Validate each stream independently.
        for stream in expected_streams:
            stream_group = (
                group.loc[group["stream"] == stream]
                .sort_values("stream_frame_index")
            )

            if len(stream_group) != expected_count:
                raise ValueError(
                    f"Run {run}, {filename}: expected "
                    f"{expected_count} {stream.upper()} records, "
                    f"found {len(stream_group)}."
                )

            actual_stream_indices = (
                stream_group["stream_frame_index"]
                .astype(int)
                .to_numpy()
            )

            expected_stream_indices = np.arange(expected_count)

            if not np.array_equal(
                actual_stream_indices,
                expected_stream_indices,
            ):
                raise ValueError(
                    f"Run {run}, {filename}: "
                    f"{stream.upper()} frame indices are "
                    "not consecutive."
                )

            expected_payload_length = EXPECTED_PAYLOAD_BYTES[stream]

            if not np.all(
                stream_group["payload_length"].to_numpy()
                == expected_payload_length
            ):
                raise ValueError(
                    f"Run {run}, {filename}: one or more "
                    f"{stream.upper()} payload lengths are incorrect."
                )

        # DONE should behave as a consecutive radar-side counter
        # within an acquisition file.
        done_values = (
            group.loc[group["stream"] == "done", "done_value"]
            .dropna()
            .astype(np.int64)
            .to_numpy()
        )

        if len(done_values) != expected_count:
            raise ValueError(
                f"Run {run}, {filename}: expected "
                f"{expected_count} decoded DONE values, "
                f"found {len(done_values)}."
            )

        if (
            len(done_values) > 1
            and not np.all(np.diff(done_values) == 1)
        ):
            raise ValueError(
                f"Run {run}, {filename}: DONE values are not "
                f"consecutive: {done_values.tolist()}"
            )

    print("Record-manifest validation passed for all files.")
    print(
        "Note: RADC, RFFT, and DONE arrival order was not "
        "required to form repeating triplets."
    )

### 1.4 Build and validate `FILE_MANIFEST`

In [ ]:
# ---------------------------------------------------------------------
# Build and validate the binary-file manifest
# ---------------------------------------------------------------------

manifest_rows = []

for run, files in BIN_FILES.items():
    for path in files:

        # Run A is fixed-position and does not use a SAR position index.
        if run == "a":
            position_index = pd.NA
            encoder_count = pd.NA
            acquisition_type = "fixed_position"

        else:
            position_index = parse_sar_position_index(path.name)

            if position_index not in SAR_POSITION_INDICES:
                raise ValueError(
                    f"Run {run.upper()} contains unexpected position "
                    f"index {position_index}: {path.name}"
                )

            encoder_count = int(
                GALIL_ENCODER_COUNTS[position_index]
            )

            acquisition_type = "sar_scan"

        manifest_rows.append(
            {
                "run": run.upper(),
                "acquisition_type": acquisition_type,
                "position_index": position_index,
                "encoder_count": encoder_count,
                "filename": path.name,
                "file_size_bytes": path.stat().st_size,
                "path": path,
            }
        )


FILE_MANIFEST = pd.DataFrame(manifest_rows)

FILE_MANIFEST["position_index"] = (
    FILE_MANIFEST["position_index"].astype("Int64")
)

FILE_MANIFEST["encoder_count"] = (
    FILE_MANIFEST["encoder_count"].astype("Int64")
)

FILE_MANIFEST = (
    FILE_MANIFEST
    .sort_values(
        ["run", "position_index"],
        na_position="first",
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# Validate the file-level manifest
# ---------------------------------------------------------------------

expected_file_counts = {
    "A": 1,
    "B": 17,
    "C": 17,
    "D": 17,
}

actual_file_counts = (
    FILE_MANIFEST.groupby("run")
    .size()
    .to_dict()
)

if actual_file_counts != expected_file_counts:
    raise ValueError(
        f"Unexpected file counts: {actual_file_counts}"
    )


expected_positions = set(
    np.asarray(SAR_POSITION_INDICES, dtype=int)
)

for run in ["B", "C", "D"]:
    run_rows = FILE_MANIFEST.loc[
        FILE_MANIFEST["run"] == run
    ]

    run_positions = set(
        run_rows["position_index"]
        .dropna()
        .astype(int)
    )

    if run_positions != expected_positions:
        raise ValueError(
            f"Run {run} positions are incorrect. "
            f"Found: {sorted(run_positions)}"
        )

    if run_rows["position_index"].duplicated().any():
        raise ValueError(
            f"Run {run} contains duplicate position indices."
        )


print("File manifest validation passed.")
print()
print("Files per run:")
print(FILE_MANIFEST.groupby("run").size())

print()
print("File sizes by run:")
print(
    FILE_MANIFEST.groupby("run")["file_size_bytes"]
    .agg(["count", "min", "max", "nunique"])
)

### 1.5 Build `RECORD_MANIFEST`
Build the permanent record-level manifest

In [ ]:
# ---------------------------------------------------------------------
# Build the record-level manifest
# ---------------------------------------------------------------------

record_manifest_parts = []

for file_row in FILE_MANIFEST.itertuples(index=False):
    records = scan_vmd3_bin(file_row.path)

    records.insert(0, "run", file_row.run)
    records.insert(
        1,
        "acquisition_type",
        file_row.acquisition_type,
    )
    records.insert(
        2,
        "position_index",
        file_row.position_index,
    )
    records.insert(
        3,
        "encoder_count",
        file_row.encoder_count,
    )
    records.insert(4, "filename", file_row.filename)
    records.insert(5, "path", file_row.path)

    records["done_value"] = pd.Series(
        pd.NA,
        index=records.index,
        dtype="Int64",
    )

    for index in records.index[
        records["stream"] == "done"
    ]:
        payload = read_record_payload(
            records.at[index, "path"],
            records.at[index, "payload_offset"],
            records.at[index, "payload_length"],
        )

        records.at[index, "done_value"] = decode_done(
            payload
        )

    record_manifest_parts.append(records)


RECORD_MANIFEST = pd.concat(
    record_manifest_parts,
    ignore_index=True,
)

RECORD_MANIFEST["position_index"] = (
    RECORD_MANIFEST["position_index"].astype("Int64")
)

RECORD_MANIFEST["encoder_count"] = (
    RECORD_MANIFEST["encoder_count"].astype("Int64")
)

RECORD_MANIFEST["done_value"] = (
    RECORD_MANIFEST["done_value"].astype("Int64")
)

print("Record manifest created successfully.")
print(f"Total stored records: {len(RECORD_MANIFEST):,}")

Validate and summarize `RECORD_MANIFEST`

In [ ]:
# ---------------------------------------------------------------------
# Validate and summarize the record-level manifest
# ---------------------------------------------------------------------

validate_record_manifest(RECORD_MANIFEST)

print()
print("Frames by run and stream:")

display(
    RECORD_MANIFEST.groupby(["run", "stream"])
    .size()
    .unstack(fill_value=0)
)

### 1.6 Load and Validate Run A Data

In [ ]:
# ---------------------------------------------------------------------
# Load and validate Run A data
# ---------------------------------------------------------------------

run_a_radc = load_radc_stack(
    RECORD_MANIFEST,
    run="A",
)

run_a_rfft = load_rfft_stack(
    RECORD_MANIFEST,
    run="A",
)

run_a_done = load_done_values(
    RECORD_MANIFEST,
    run="A",
)


# Confirm that all three streams contain the same number of frames.
n_radc_frames = run_a_radc.shape[0]
n_rfft_frames = run_a_rfft.shape[0]
n_done_frames = len(run_a_done)

if not (
    n_radc_frames
    == n_rfft_frames
    == n_done_frames
):
    raise ValueError(
        "Run A stream counts do not match:\n"
        f"  RADC: {n_radc_frames}\n"
        f"  RFFT: {n_rfft_frames}\n"
        f"  DONE: {n_done_frames}"
    )


# Validate the decoded complex-data stacks.
validate_complex_stack(
    run_a_radc,
    expected_frame_shape=RADC_FRAME_SHAPE,
    expected_frames=n_radc_frames,
    label="Run A RADC",
)

print()

validate_complex_stack(
    run_a_rfft,
    expected_frame_shape=RFFT_FRAME_SHAPE,
    expected_frames=n_rfft_frames,
    label="Run A FPGA RFFT",
)


# Confirm continuity of the radar-side DONE counter.
done_differences = np.diff(
    run_a_done.astype(np.int64)
)

if not np.all(done_differences == 1):
    raise ValueError(
        "Run A DONE values are not consecutive: "
        f"{run_a_done.tolist()}"
    )


print()
print("Run A stream pairing checks passed.")
print(f"  Number of captures: {n_radc_frames}")
print(f"  DONE values:        {run_a_done.tolist()}")
print()
print("Run A array conventions:")
print(
    "  run_a_radc:",
    run_a_radc.shape,
    "= (frame, sample, chirp, channel)",
)
print(
    "  run_a_rfft:",
    run_a_rfft.shape,
    "= (frame, range_bin, slow_time_index, channel)",
)

# Current Cell Development Debug Below